# Reviewer check: sensitivity of the LOCO results to the three cohorts that are never held out

Reviewer 1, comment 5 (cohort structure). AtezoTRIBE and COLOBIOME (CRC samples only) and NSHII (448 controls, 14 CRC) are in the training set of every fold and together carry near-degenerate label distributions (28–35 % of each training set). `run_loco.py --exclude-training-only` repeats the whole LOCO pipeline with these three cohorts removed from every training set (`../M2_loco_crc_pipeline/results_sensitivity/`). This notebook compares the two runs: AUCs, consensus pairs and the stable pairs (same rule as `stable_pairs.ipynb`).


In [ ]:
import os, numpy as np, pandas as pd
MAIN, SENS = '../M2_loco_crc_pipeline/results', '../M2_loco_crc_pipeline/results_sensitivity'
numbers = []
def note(claim, value, source): numbers.append({'section': 'R5', 'claim': claim, 'value': value, 'source': source}); print(f'{claim}: {value}')
rows = []
for dt in ('taxa', 'functions', 'combined'):
    for lab, d in (('main', MAIN), ('no single-class / training-only cohorts', SENS)):
        p = f'{d}/{dt}/loco_results_{dt}.csv'
        if not os.path.exists(p): continue
        r = pd.read_csv(p).set_index('Test Cohort')
        for sc in ('All Features', 'Synthetic Only', 'All Features + Synthetic'):
            rows.append({'data_type': dt, 'training': lab, 'scenario': sc, 'mean_AUC': r.loc['Mean', sc], 'weighted_AUC': r.loc['Weighted Mean', sc]})
auc = pd.DataFrame(rows).pivot_table(index=['data_type', 'scenario'], columns='training', values=['mean_AUC', 'weighted_AUC'])
auc.to_csv('sensitivity_auc.tsv', sep='\t'); auc.round(3)


In [ ]:
# stable pairs under both runs (>= 80 % of folds and above both constituents)
def stable(d):
    out = []
    for dt in ('taxa', 'functions', 'combined'):
        raw = pd.read_csv(f'{d}/{dt}/base_features_{dt}.csv').set_index('Feature')['Selection %']
        syn = pd.read_csv(f'{d}/{dt}/synthetic_features_{dt}.csv').sort_values('Selection %', ascending=False)
        seen = set()
        for _, r in syn.iterrows():
            name = r['Feature'][3:] if r['Feature'][:3] in ('LR_', 'GM_') else r['Feature']
            if '__' not in name or (dt, name) in seen: continue
            seen.add((dt, name)); f1, f2 = name.split('__', 1)
            if r['Selection %'] >= 80 and r['Selection %'] > raw.get(f1, 0) and r['Selection %'] > raw.get(f2, 0):
                out.append((dt, name))
    return set(out)
sm_, ss_ = stable(MAIN), stable(SENS)
for dt in ('taxa', 'functions', 'combined'):
    a, b = {k for k in sm_ if k[0] == dt}, {k for k in ss_ if k[0] == dt}
    note(f'{dt}: stable pairs main / sensitivity run / shared (Jaccard)', f'{len(a)} / {len(b)} / {len(a & b)} ({len(a & b) / len(a | b):.2f})', 'R5 sensitivity_training_cohorts.ipynb')
for dt in ('taxa', 'functions', 'combined'):
    for lab, d in (('main', MAIN), ('sensitivity', SENS)):
        r = pd.read_csv(f'{d}/{dt}/loco_results_{dt}.csv').set_index('Test Cohort')
        note(f'{dt} mean AUC ({lab}): raw / synthetic / raw+synthetic', ' / '.join(f"{r.loc['Mean', s]:.3f}" for s in ('All Features', 'Synthetic Only', 'All Features + Synthetic')), f'{d}/{dt}/loco_results_{dt}.csv')
hub = lambda S, f: sum(f in k[1] for k in S if k[0] == 'combined')
note('P. micra / F. nucleatum stable combined pairs, main vs sensitivity', f"{hub(sm_, 'Parvimonas_micra')}/{hub(sm_, 'Fusobacterium_nucleatum')} vs {hub(ss_, 'Parvimonas_micra')}/{hub(ss_, 'Fusobacterium_nucleatum')}", 'R5 sensitivity_training_cohorts.ipynb')
pd.DataFrame(sorted(sm_ & ss_), columns=['data_type', 'pair']).to_csv('sensitivity_shared_stable_pairs.tsv', sep='\t', index=False)
pd.DataFrame(numbers).to_csv('r5_numbers_sensitivity.tsv', sep='\t', index=False)
